# Piano B — Notebook K: confronti aggiuntivi per la tesi

Coda `K` (40 job), in ordine di priorità:
1. **Riconoscimento della clip**: quale delle 28 clip si sta guardando, su soggetti mai visti (split FM).
   DE + z-score, EEGNet + EA, TSception + EA (4 s e 10 s), CBraMod. Serve alla figura F10.
2. **Protocollo entro soggetto** (`split_fm_within`): i soggetti di test entrano nel training con 2 dei 3
   segmenti da 10 s di ogni clip e si testano sul terzo, come in molta letteratura. Figura F9.
3. **Lunghezza della finestra** (2 e 4 s contro 10 s) nel leave-clip-out: priorità bassa. Figura F13.

Impostazioni: Accelerator **GPU T4 x2**, Internet off. Input: `pianob-faced-data`, `pianob-code`, i pesi di
CBraMod (come per i notebook B, C, F) e, per riprendere, l'output della sessione precedente di questo notebook.

In [ ]:
import os
os.environ["PIANOB_ACCOUNT"] = "nbK_confronti"   # la ripresa usa solo gli output di questo notebook

In [ ]:
# Setup: trova il codice (Dataset "pianob-code"), lo copia fuori da /kaggle/working (così non finisce
# negli output) e lo aggiunge a sys.path. Non serve modificare nulla qui.
import glob, os, shutil, sys, time
os.environ.setdefault("PIANOB_T0", str(time.time()))      # inizio della sessione (guardia delle 12 h)
os.environ.setdefault("PIANOB_BUDGET_H", "11.3")           # stop pulito prima del limite Kaggle
SYNTHETIC = os.environ.get("PIANOB_SYNTHETIC") == "1"      # solo per prove locali


def _find_code():
    cands = []
    roots = ["/kaggle/input"] if os.path.exists("/kaggle/input") else []
    roots += [os.getcwd(), os.path.dirname(os.getcwd())]
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", "pianob", "__init__.py"), recursive=True):
            base = os.path.dirname(os.path.dirname(p))
            if "pianob_state" not in base:
                cands.append(base)
    assert cands, "Codice non trovato: aggiungi il Dataset 'pianob-code' come input (Add Input)."
    # se ci sono più copie, usa quella con VERSION più recente
    def _ver(b):
        v = os.path.join(b, "VERSION")
        return open(v).read().strip() if os.path.exists(v) else ""
    return sorted(cands, key=_ver)[-1]


CODE_SRC = _find_code()
if os.path.exists("/kaggle/working"):
    CODE = "/tmp/pianob_code"
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(CODE_SRC, CODE, ignore=shutil.ignore_patterns("notebooks", "*.zip", "local_work"))
else:
    CODE = CODE_SRC
sys.path.insert(0, CODE)
os.chdir(CODE)
print("codice da", CODE_SRC, "| versione", open(os.path.join(CODE, "VERSION")).read().strip()
      if os.path.exists(os.path.join(CODE, "VERSION")) else "(locale)")

from pianob import kaggle_env, nb_helpers  # noqa: E402
nb_helpers.print_env()
kaggle_env.restore_state_from_inputs()

In [ ]:
ACCOUNT = os.environ["PIANOB_ACCOUNT"]
nb_helpers.weights_check()
nb_helpers.run_queue("K", gpus="auto", synthetic=SYNTHETIC, cpu_workers=2)
nb_helpers.queue_status("K")

In [ ]:
nb_helpers.finish(ACCOUNT)